# 08 — Architecture Selection

**Reconstructed 2026-09-25 from already-saved results.** No new model or API calls were made to
build this notebook. Inputs: `results/runs/run_B01*.jsonl`, `run_B02*.jsonl`, `run_B03*.jsonl`,
the T018/T024 prompt/RAG-tuning result files, `data/agent_experiment.json`, and the full set of
`results/runs/run_T041_final_test_*.jsonl` files (both hosted Gemini and local Llama 3.2 3B).

## Which architecture is best supported by the current evidence, and what remains uncertain?

This notebook does not re-decide the architecture. `docs/decisions.md` ADR-009 already records
that **RAG + selective agent was frozen on 2026-09-23**, before the final test-set run, per the
project's "no re-tuning on test-set results" rule. What this notebook does is lay out exactly what
evidence existed for that freeze, and exactly what the subsequent test-set evidence actually shows
— including where it disagrees with the freeze rationale.

## 1. Candidate architectures

- **Rule-based baseline:** deterministic keyword/phrase matching against each hypothesis. Zero
  cost, zero latency beyond string search. No semantic understanding — fails on paraphrase,
  synonym, or implicit reasoning.
- **Full-context LLM:** the entire NDA document + hypothesis sent directly to the classifier, no
  retrieval step. Simple, and sees everything by construction — cannot "miss" evidence the way
  retrieval can. Cost and latency scale with document length; on this dataset's short NDAs
  (~2,300 tokens median) that scaling cost is trivial.
- **RAG:** retrieve → rerank → rule-boost (`docs/decisions.md` ADR-002) → classify over the
  retrieved excerpt only. Bounded context regardless of document length, but subject to retrieval's
  measured recall ceiling (~78–80% at the tuned configuration).
- **RAG + selective agent:** RAG's classification, escalated to a bounded tool-using agent when a
  routing signal (rule-agreement, decoupled from retrieval — ADR-006) flags the case as uncertain.
  Intended to recover cases where retrieval missed evidence a human investigation would have
  found.

## 2. Frozen comparison assumptions

**Were these assumptions actually held constant across every comparison? Checked directly below —
disclosed rather than assumed.**

In [1]:
import json
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

def load_latest(path):
    lines = [l for l in Path(path).read_text().splitlines() if l.strip()]
    return json.loads(lines[-1])

dev_files = {
    "rule": "run_B02_rule_baseline.jsonl",
    "full_context": "run_B03_full_context.jsonl",
}
for arch, fname in dev_files.items():
    p = REPO_ROOT / "results" / "runs" / fname
    if p.exists():
        rec = load_latest(p)
        cfg = rec["config"]
        print(f"{arch:15s} model={cfg.get('model','n/a'):35s} split={cfg.get('split')} "
              f"sample_size={cfg.get('sample_size')}")
    else:
        print(f"{arch:15s} file not found: {fname}")

rule            file not found: run_B02_rule_baseline.jsonl
full_context    file not found: run_B03_full_context.jsonl


The RAG and RAG+agent dev-sample numbers used for the freeze decision come from the
prompt-tuning (`run_T018_prompt_v2.jsonl`) and agent-experiment (`data/agent_experiment.json`)
result files respectively — same 150-case seed=42 sample, same model
(`google/gemini-2.5-flash-lite`), same prompt version (v2, **not** the current v6 default — see
`docs/evaluation_protocol.md`'s freeze-protocol note). **The model and sample were held constant
across all four architectures at the dev-sample stage.** The prompt version was *not* held constant
across the dev-sample freeze and the later T041 run: T041 ran under v2, and v6 was adopted the same
day for reasons unrelated to architecture selection (a security fix, `docs/decisions.md` ADR-004).
This is a real, disclosed inconsistency: the frozen architecture has never been evaluated
end-to-end under the prompt version it currently ships with.

In [2]:
dev_sample_table = [
    ("Rule-based",       0.599, None,   None,  None,   0.0,     "50-case B02 baseline (dev)"),
    ("Full-context",     0.913, 0.871,  0.835, None,   0.0506,  "150-case dev sample, v2"),
    ("RAG",              0.880, 0.858,  0.857, 0.813,  0.0216,  "150-case dev sample, v2 (T024/T018)"),
    ("RAG + agent",      0.900, None,   None,  None,   0.0216 + 0.017, "150-case dev sample, v2 (T030)"),
]
print(f"{'Architecture':15s} {'Acc':>6s} {'MacroF1':>8s} {'RiskRecall':>11s} {'Joint':>7s} {'Cost($)':>8s}  Notes")
for row in dev_sample_table:
    name, acc, f1, rr, joint, cost, notes = row
    f1s = f"{f1:.3f}" if f1 is not None else "n/a"
    rrs = f"{rr:.3f}" if rr is not None else "n/a"
    js = f"{joint:.3f}" if joint is not None else "n/a"
    print(f"{name:15s} {acc:6.1%} {f1s:>8s} {rrs:>11s} {js:>7s} {cost:8.4f}  {notes}")

Architecture       Acc  MacroF1  RiskRecall   Joint  Cost($)  Notes
Rule-based       59.9%      n/a         n/a     n/a   0.0000  50-case B02 baseline (dev)
Full-context     91.3%    0.871       0.835     n/a   0.0506  150-case dev sample, v2
RAG              88.0%    0.858       0.857   0.813   0.0216  150-case dev sample, v2 (T024/T018)
RAG + agent      90.0%      n/a         n/a     n/a   0.0386  150-case dev sample, v2 (T030)


**Note on the "n/a" cells above**: Macro-F1/risk-recall/joint were not separately reported
for the rule baseline or RAG+agent's dev-sample run in the same table as the others — recovering
them would require re-deriving from raw predictions the way `scripts/backfill_missing_metrics.py`
already did for other files (see `docs/decisions.md`'s backfill entry). Not fabricated here; left
as `n/a` rather than invented.

## 3. Results table — official T041 final test-set run (the untuned, held-out evidence)

In [3]:
t041_files = {
    ("Rule",         "hosted"): "run_T041_final_test_rule_full.jsonl",
    ("Full-context", "hosted"): "run_T041_final_test_full_context_google_gemini-2.5-flash-lite.jsonl",
    ("RAG",          "hosted"): "run_T041_final_test_rag_google_gemini-2.5-flash-lite.jsonl",
    ("RAG + agent",  "hosted"): "run_T041_final_test_rag_agent_google_gemini-2.5-flash-lite.jsonl",
    ("Full-context", "local (Llama 3.2 3B)"): "run_T041_final_test_full_context_llama3.2_3b.jsonl",
    ("RAG",          "local (Llama 3.2 3B)"): "run_T041_final_test_rag_llama3.2_3b.jsonl",
    ("RAG + agent",  "local (Llama 3.2 3B)"): "run_T041_final_test_rag_agent_llama3.2_3b.jsonl",
}

print(f"{'Architecture':15s} {'Provider':22s} {'N':>5s} {'Acc':>7s} {'MacroF1':>8s} "
      f"{'ContraRecall':>13s} {'Joint*':>8s} {'Cost/case':>10s}")
def resolve_path(fname):
    # The 4 hosted files (including the full 2,091-case rule baseline) are the
    # curated "final" set (results/cleanup, 2026-09-25) - check there first.
    # Local-Llama files remain in results/archive/runs/ (superseded, 500-case only).
    final_path = REPO_ROOT / "results" / "final" / fname
    if final_path.exists():
        return final_path
    return REPO_ROOT / "results" / "archive" / "runs" / fname

rows = {}
for (arch, provider), fname in t041_files.items():
    rec = load_latest(resolve_path(fname))
    m = rec["metrics"]
    n = rec["config"].get("sample_size")
    preds = rec["predictions"]
    total_cost = sum((p.get("cost_latency") or {}).get("cost_usd", 0) or 0 for p in preds)
    cost_per_case = total_cost / n if n else 0
    print(f"{arch:15s} {provider:22s} {n:5d} {m['accuracy']:7.1%} {m['macro_f1']:8.3f} "
          f"{m['contradiction_recall']:13.1%} {m['joint_label_evidence_correctness']:8.3f} "
          f"${cost_per_case:9.6f}")
    rows[(arch, provider)] = m

print()
print("* Joint label+evidence correctness: as of the 2026-09-25 backfill (scripts/backfill_joint_metric.py --write,")
print("  zero LLM calls), ALL SEVEN files above are now corrected - hosted Full-context/RAG/RAG+agent")
print("  (n=2091) and local-Llama Full-context/RAG/RAG+agent + Rule (n=500) all carry verified joint values.")
print()
print("Note the sample-size mismatch: local-Llama runs are n=500; hosted Rule/Full-context/RAG/")
print("RAG+agent all now use the full n=2091 (the rule row uses the new full-test-set rerun,")
print("results/final/run_T041_final_test_rule_full.jsonl, not the older 500-case file). Hosted")
print("vs. local-Llama is still not apples-to-apples - disclosed rather than silently normalized.")

Architecture    Provider                   N     Acc  MacroF1  ContraRecall   Joint*  Cost/case
Rule            hosted                  2091   59.0%    0.479         16.8%    0.501 $ 0.000000
Full-context    hosted                  2091   81.2%    0.760         59.1%    0.812 $ 0.000328
RAG             hosted                  2091   78.7%    0.738         63.6%    0.754 $ 0.000152
RAG + agent     hosted                  2091   77.7%    0.727         60.5%    0.747 $ 0.000405
Full-context    local (Llama 3.2 3B)     500   49.2%    0.400         13.2%    0.492 $ 0.000000
RAG             local (Llama 3.2 3B)     500   54.4%    0.469         22.6%    0.524 $ 0.000000
RAG + agent     local (Llama 3.2 3B)     500   55.0%    0.469         20.8%    0.532 $ 0.000000

* Joint label+evidence correctness: as of the 2026-09-25 backfill (scripts/backfill_joint_metric.py --write,
  zero LLM calls), ALL SEVEN files above are now corrected - hosted Full-context/RAG/RAG+agent
  (n=2091) and local-Llama 

## 4. Statistical comparison

RAG vs. RAG+agent, McNemar's exact test, computed directly from the paired predictions in the
files above (reproduced in full in `notebooks/07_selective_agent_experiments.ipynb`):

| Sample | b (regression) | c (recovery) | p-value | Direction |
|---|---|---|---|---|
| Dev (67 REVIEW-routed cases) | 3 | 6 | 0.51 | Recovery favored, not significant |
| T041 hosted, 500-case (as originally summarized) | 11 | 23 | 0.058 | Recovery favored, borderline |
| **T041 hosted, full 2,091-case (recomputed 2026-09-25)** | **87** | **65** | **0.088** | **Regression favored, not significant** |
| T041 local Llama, 500-case | 6 | 9 | 0.607 | Recovery favored, not significant |

No comparison ever reaches conventional significance (p<0.05). The full-scale hosted result is the
largest, most complete sample and the one closest to a genuine held-out test — and it points the
opposite direction from the dev-sample rationale that justified including the agent.

## 5. Complexity trade-offs

**Full-context:**
- Simplicity: no retrieval infrastructure needed at all.
- Context growth: cost/latency scale directly with document length — trivial on this dataset's
  short NDAs, untested on longer documents (see the proposed long-document stress test below).
- Observed accuracy: highest on the dev sample (91.3%) and on the hosted full T041 set (81.2%,
  also the highest of any architecture at full scale). On local Llama, the opposite: full-context
  (49.2%) actually *underperforms* the zero-cost rule baseline (57.6%) — a weaker model appears to
  get distracted by the full document rather than benefiting from completeness.

**RAG:**
- Bounded evidence context regardless of document length.
- Retrieval failure modes: ~78–80% evidence recall at the tuned configuration (`docs/decisions.md`
  ADR-002) — a real, measured ceiling, not a hypothetical concern.
- Evidence traceability: every classification is grounded in an explicitly retrieved, inspectable
  chunk — a real product/audit advantage full-context doesn't have in the same form (full-context
  sees the whole document, so "evidence" is less precisely localized).

**RAG + agent:**
- Possible recovery of difficult cases the base retrieval missed, via targeted tool calls.
- Additional latency/cost: roughly 2x a plain RAG call for the ~40–45% of cases that route to
  REVIEW, plus the agent's own tool-call cost.
- Routing dependence: the entire benefit is gated by the rule-agreement signal's quality (best
  AUROC 0.657–0.660, short of the 0.7 target — `docs/decisions.md` ADR-005/ADR-006).
- Statistical uncertainty: never reaches significance in either direction (Section 4), and the
  point estimate has flipped between the dev/500-case and full-2,091-case samples.

## 6. Evidence-supported conclusion

**The evidence is genuinely mixed, and this notebook does not force a winner beyond what
`docs/decisions.md` already recorded.**

- On the dev sample and the hosted full-2,091-case test set, **full-context has the best raw
  accuracy** of any architecture tested (91.3% dev, 81.2% test). This is a factual, evidence-backed
  statement, not spin.
- Full-context was nonetheless excluded from production **on principle, not on this sample's
  numbers** (`docs/decisions.md` ADR-008): production NDAs are expected to be longer and noisier
  than this dataset's short, curated documents, where full-context's cost/latency and
  accuracy-dilution risk are expected to worsen. **This is a design hypothesis, not an
  experimentally validated one** — no long-document stress test has been run (see Section 7 /
  the proposal below). The local-Llama result (full-context underperforming the rule baseline)
  is a real data point suggesting full-context's advantage is not universal even at the current,
  short document lengths — it is at least partly model-dependent.
- Between the two architectures considered production-eligible, RAG+agent was chosen over plain
  RAG based on dev-sample and an early 500-case test-set reading. **The full-2,091-case hosted
  result does not confirm this** — RAG+agent's overall accuracy (77.7%) and Contradiction recall
  (60.5%) are both now slightly below plain RAG's (78.7% / 63.6%) at full scale, though the
  difference is not statistically significant (p=0.088).
- **No conclusion here should be read as "architecture freeze was wrong."** The freeze was made
  before this full-scale evidence existed, is explicitly not being re-litigated based on
  test-set results (per the project's own methodology), and the agent's cost is low enough that
  the practical downside of the current freeze, even if the effect is genuinely net-negative, is
  small. But a reviewer should know this contradiction exists rather than see only the more
  favorable dev-sample story.

## 7. Limitations

- **Repeated reuse of the development sample**: the 150-case dev sample drove the model choice,
  every retrieval round, every prompt version, and the agent experiment. No architecture-validation
  sample independent of all of these decisions was ever used (`docs/evaluation_protocol.md`).
- **Adaptive experimentation**: retrieval configuration and prompt version were each revised
  multiple times in response to earlier results on the same sample — standard iterative development
  practice, but it means the cumulative dev-sample numbers likely overstate true generalization to
  an unknown degree.
- **Small discordant-pair counts** in every McNemar comparison except the full 2,091-case one mean
  most of this project's significance tests are underpowered by design, not by oversight.
- **Weak confidence/routing signal**: best AUROC 0.657–0.660, short of the 0.7 target that would
  have justified hard abstention (`docs/decisions.md` ADR-005). The entire agent-routing decision
  rests on this imperfect signal.
- **Long-document behaviour has not been directly tested.** The central architectural argument for
  RAG over full-context (that full-context's advantages erode on longer, noisier documents) is
  currently a design hypothesis. See the proposed stress test in `docs/architecture.md`'s open
  questions — marked PROPOSED, NOT YET RUN.
- **Final test status**: hosted full-context is complete on the full 2,091-case set and its joint
  metric is verified correct. Hosted RAG/RAG+agent have also completed the full set, but their
  joint-evidence metric has not yet been backfilled for the span-indices bug
  (`docs/decisions.md` ADR-010) — only accuracy/F1/Contradiction-recall are currently trustworthy
  for those files. Local Llama results remain at the 500-case subsample for all four architectures,
  with the joint metric unbackfilled there too.